# Testing coding and testing agents for human eval dataset

## AgentCoder Programmer Prompt

In [13]:
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_ollama.chat_models import ChatOllama
from langgraph.graph import StateGraph, START, END
from typing import List, TypedDict, Dict
import common

# initialize code processor
code_processor = common.CodeProcessor()

# Initialize the Ollama model
llm = ChatOllama(model="qwen2.5-coder:7b")

# 1. State with separate memory "sub-folders"
class AgentState(TypedDict):
    problem: str # The coding problem from the HumanEval dataset
    solution: str # The solution code generated by the programmer agent
    tests: str # The tests generated by the test designer agent

    # agent_memories is the "shared project folder"
    agent_memories: Dict[str, List]

# System prompts
PROGRAMMER_SYSTEM_PROMPT = "You are an expert software developer."
TEST_DESIGNER_SYSTEM_PROMPT = "You are an expert program tester."

# Initial Prompts
PROGRAMMER_PROBLEM_PROMPT = """As a programmer, you are required to complete the function. Use a Chain-of-Thought approach to break down the problem, create pseudocode, and then write the code in Python language. Ensure that your code is efficient, readable, and well-commented.  

For example:  
**Input Code Snippet**: 
```python 
{problem_placeholder}
# Add your code here to complete the function 
```  
**Instructions**: 
1. **Understand and Clarify**: Make sure you understand the task. 
2. **Algorithm/Method Selection**: Decide on the most efficient way. 
3. **Pseudocode Creation**: Write down the steps you will follow in pseudocode. 
4. **Code Generation**: Translate your pseudocode into executable Python code."""

TEST_DESIGNER_PROBLEM_PROMPT = """As a tester, your task is to create comprehensive test cases for the incomplete `{function_name}` function. These test cases should encompass Basic, Edge, and Large Scale scenarios to ensure the code's robustness, reliability, and scalability.

**Input Code Snippet**: 
```python 
{problem_placeholder}
```

**1. Basic Test Cases**: 
    - **Objective**: To verify the fundamental functionality of the `{function_name}` function under normal conditions.  
**2. Edge Test Cases**: 
    - **Objective**: To evaluate the function's behavior under extreme or unusual conditions.  
**3. Large Scale Test Cases**: 
    - **Objective**: To assess the function's performance and scalability with large data samples.  

**Instructions**: 
    - Use the `unittest` framework for writing the test cases.
    - Implement a comprehensive set of test cases following the guidelines above. 
    - Ensure each test case is well-documented with comments explaining the scenario it covers. 
    - Pay special attention to edge cases as they often reveal hidden bugs. 
    - For large-scale tests, focus on the function's efficiency and performance under heavy loads.
    - Do not implement the function, another expert programmer will do that.
"""

# 2. Agents coded to access ONLY their own memory
def programmer_agent(state: AgentState) -> AgentState:
    """Programmer agent"""    
    # This agent ONLY gets its memory from the 'programmer' key
    programmer_memory = state['agent_memories'].get('programmer', [])
    
    new_prompt = "" # Placeholder, not used in current implementation

    messages = [SystemMessage(content=PROGRAMMER_SYSTEM_PROMPT)] + [HumanMessage(content=PROGRAMMER_PROBLEM_PROMPT.format(problem_placeholder=state['problem']))] + programmer_memory + [new_prompt]
    
    response = llm.invoke(messages)
    solution = code_processor.extract_code_block_from_response(response.content)
    
    state['solution'] = solution
    # It ONLY writes back to its own memory
    state['agent_memories']['programmer'] = programmer_memory + [new_prompt, AIMessage(content=solution)]
    
    return state

def test_designer_agent(state: AgentState) -> AgentState:
    """Test designer agent"""
        
    # This agent ONLY gets its memory from the 'test_designer' key
    test_designer_memory = state['agent_memories'].get('test_designer', [])
    
    function_name = code_processor.extract_function_name_from_problem(state['problem'])

    # Note: It can access other parts of the state like the 'solution' if needed
    new_prompt = "" # Placeholder, not used in current implementation
    messages = [SystemMessage(content=TEST_DESIGNER_SYSTEM_PROMPT)] + [HumanMessage(content=TEST_DESIGNER_PROBLEM_PROMPT.format(function_name=function_name, problem_placeholder=state['problem']))] + test_designer_memory + [new_prompt]
    
    response = llm.invoke(messages)
    tests = response.content

    state['tests'] = tests


    # It ONLY writes back to its own memory
    state['agent_memories']['test_designer'] = test_designer_memory + [new_prompt, AIMessage(content=tests)]
    return state

# 3. Build and run the graph
graph = StateGraph(AgentState)
graph.add_node("programmer", programmer_agent)
graph.add_node("test_designer", test_designer_agent)

graph.add_edge(START, "programmer")
graph.add_edge("programmer", "test_designer")
graph.add_edge("test_designer", END)

# Compile the graph
agent_coder = graph.compile()

# Example problem for testing
sample_problem = '''def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """ Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """'''

print("🚀 Starting Multi-Agent Code Generation and Testing System")
print("=" * 60)

# Initialize state
initial_state = {
    "problem": sample_problem,
    "solution": "",
    "tests": "",
    "agent_memories": {"programmer": [], "test_designer": []}
}

# Run the multi-agent workflow
result = agent_coder.invoke(initial_state)

print("=" * 60)
print("✅ Multi-Agent Workflow Complete!")
print(f"📝 Final Solution Length: {len(result['solution'])} characters")
print(f"🧪 Final Tests Length: {len(result['tests'])} characters")


print("\n--- Generated Solution ---\n")
print(result['solution'])

print("\n--- Generated Tests ---\n")
print(result['tests'])

🚀 Starting Multi-Agent Code Generation and Testing System
✅ Multi-Agent Workflow Complete!
📝 Final Solution Length: 679 characters
🧪 Final Tests Length: 1617 characters

--- Generated Solution ---

def has_close_elements(numbers: List[float], threshold: float) -> bool:
    """
    Check if in given list of numbers, are any two numbers closer to each other than
    given threshold.
    >>> has_close_elements([1.0, 2.0, 3.0], 0.5)
    False
    >>> has_close_elements([1.0, 2.8, 3.0, 4.0, 5.0, 2.0], 0.3)
    True
    """
    # Sort the list of numbers 
    numbers.sort()

    # Iterate through the sorted list (except the last element)
    for i in range(len(numbers) - 1):
        # Compare current number with its next neighbor
        if abs(numbers[i] - numbers[i + 1]) < threshold:
            return True

    # If no such pair is found, return False
    return False

--- Generated Tests ---

```python
import unittest
from typing import List

def has_close_elements(numbers: List[float], 